# Fine-Tuning Road Damage (RDD2022) & Indian Traffic Sign Models on Kaggle GPU

## 1. Attach These Kaggle Datasets via **+ Add Input**
1. `aliabdelmenam/rdd-2022` (Road Damage 2022)
2. `redzapdos123/indian-driving-dataset-detections-yolov11` (IDD — used for **cross-domain vehicle hard negatives**)
3. `kaustubhrastogi17/traffic-signs-dataset-indian-roads` (Indian Traffic Signs)
4. `andrewmvd/pothole-detection` *(Recommended 4th input — auto-detected & merged into class `4 (pothole)`, supports both VOC XML and YOLO `.txt`)*

## 2. Kaggle Notebook Settings
- **Accelerator**: `GPU T4 x2` or `GPU P100`
- **Internet**: `ON` (to clone your repository)

## 3. Domain-Specific Fixes Applied Automatically
- **Traffic Sign**: Disables horizontal flip (`fliplr: 0.0` so directional arrows, STOP text, and speed digits are never mirrored), preserves sign color semantics (`hsv_s: 0.35`), adds **22% IDD vehicle hard-negative images** (empty `.txt` labels) so the detector natively stops predicting vehicles as traffic signs, and fine-tunes at `1280px` in 2 stages (`--freeze 10` -> `--freeze 0`).
- **Road Damage (RDD2022)**: Upgrades resolution (`640` -> `896px`) with `--multi-scale`, widens anchor ratio tolerance (`anchor_t: 5.0`) for long thin cracks, enables focal loss (`fl_gamma: 1.5`) + sharper localization (`box: 0.075`), merges optional extra pothole datasets, and adds **10% IDD hard negatives** so cracks stop firing on vehicles or sky.


In [ ]:
# Cell 1: Check GPU, Clone Main Repo, Clone Official YOLOv5 & Apply Custom Overlays
!nvidia-smi
import os, subprocess, sys
from pathlib import Path

REPO_URL = "https://github.com/parthiv-wxyz/multi-road-object-detection.git"
WORK_DIR = Path("/kaggle/working/multi-road-object-detection")

if not WORK_DIR.exists():
    subprocess.run(["git", "clone", REPO_URL, str(WORK_DIR)], check=True)
os.chdir(WORK_DIR)
print("Working directory:", Path.cwd())
!pip install -q ultralytics ultralytics-thop PyYAML tqdm thop

# Clone official ultralytics/yolov5 into yolov5/ and overlay model_changes/ (P2, LWC3, ECA, AFP, Wise-IoU, hyps & best.pt)
subprocess.run([sys.executable, "scripts/sync_yolov5_overlay.py", "--apply"], check=True)


: 

In [ ]:
# Cell 2: Auto-Discover Mounted Kaggle Datasets under /kaggle/input (supports both flat and /kaggle/input/datasets/<owner>/<slug>)
from pathlib import Path

INPUT_ROOT = Path("/kaggle/input")
candidates = []
if INPUT_ROOT.exists():
    for p1 in sorted(INPUT_ROOT.iterdir()):
        if not p1.is_dir():
            continue
        if p1.name == "datasets":
            # New Kaggle layout: /kaggle/input/datasets/<owner>/<slug>
            for owner in sorted(p1.iterdir()):
                if owner.is_dir():
                    for slug in sorted(owner.iterdir()):
                        if slug.is_dir():
                            candidates.append(slug)
        else:
            candidates.append(p1)
            # Also inspect one level deeper in case mounted under /kaggle/input/<owner>/<slug>
            for sub in sorted(p1.iterdir()):
                if sub.is_dir():
                    candidates.append(sub)

print("Discovered Kaggle input directories:")
for p in candidates:
    print("  -", p)

def find_dataset(keywords):
    for p in candidates:
        text = str(p).lower()
        if any(k in text for k in keywords):
            return p
    return None

IDD_DIR = find_dataset(["indian-driving", "idd", "redzapdos"])
SIGN_DIR = find_dataset(["traffic-sign", "traffic_sign", "kaustubh"])
RDD_DIR = find_dataset(["rdd-2022", "rdd2022", "road-damage", "aliabdelmenam"])
EXTRA_POTHOLE_DIR = find_dataset(["pothole", "andrewmvd", "chitholian"])

print("\nResolved dataset paths:")
print("  IDD_DIR          :", IDD_DIR)
print("  SIGN_DIR         :", SIGN_DIR)
print("  RDD_DIR          :", RDD_DIR)
print("  EXTRA_POTHOLE_DIR:", EXTRA_POTHOLE_DIR)

assert SIGN_DIR is not None or RDD_DIR is not None, "Please attach RDD-2022 and/or Indian Traffic Signs via + Add Input!"
if IDD_DIR is None:
    print("WARNING: IDD dataset not found — proceeding WITHOUT IDD vehicle hard negatives.")


In [ ]:
# Cell 3: Write Domain-Tuned Hyperparameter YAMLs into yolov5/data/hyps/
from pathlib import Path

HYPS_DIR = Path("yolov5/data/hyps")
HYPS_DIR.mkdir(parents=True, exist_ok=True)

TS_HYP = """lr0: 0.0015
lrf: 0.05
momentum: 0.937
weight_decay: 0.0005
warmup_epochs: 2.0
warmup_momentum: 0.8
warmup_bias_lr: 0.05
box: 0.06
cls: 0.60
cls_pw: 1.0
obj: 1.25
obj_pw: 1.0
iou_t: 0.20
anchor_t: 4.0
fl_gamma: 0.5
hsv_h: 0.008
hsv_s: 0.35
hsv_v: 0.35
degrees: 4.0
translate: 0.10
scale: 0.45
shear: 1.0
perspective: 0.0003
flipud: 0.0
fliplr: 0.0
mosaic: 0.75
mixup: 0.10
copy_paste: 0.0
"""

RDD_HYP = """lr0: 0.0020
lrf: 0.05
momentum: 0.937
weight_decay: 0.0005
warmup_epochs: 2.0
warmup_momentum: 0.8
warmup_bias_lr: 0.05
box: 0.075
cls: 0.55
cls_pw: 1.20
obj: 1.15
obj_pw: 1.10
iou_t: 0.20
anchor_t: 5.0
fl_gamma: 1.5
hsv_h: 0.010
hsv_s: 0.45
hsv_v: 0.45
degrees: 5.0
translate: 0.12
scale: 0.55
shear: 1.5
perspective: 0.0005
flipud: 0.0
fliplr: 0.5
mosaic: 0.80
mixup: 0.15
copy_paste: 0.0
"""

(HYPS_DIR / "hyp.traffic_sign_finetune.yaml").write_text(TS_HYP)
(HYPS_DIR / "hyp.road_damage_finetune.yaml").write_text(RDD_HYP)
print("Wrote domain-tuned hyperparameter YAMLs successfully.")


In [ ]:
# Cell 4: Build Tuned Datasets with IDD Vehicle Hard Negatives & Optional Extra Pothole Merge
# NOTE: Push your latest local repo commits first so scripts/prepare_tuned_datasets.py is present!
import subprocess, sys
from pathlib import Path

PREP_ROOT = Path("/kaggle/working/prepared_tuned_datasets")
cmd = [
    sys.executable, "scripts/prepare_tuned_datasets.py",
    "--output-root", str(PREP_ROOT),
    "--sign-neg-ratio", "0.22",
    "--rdd-neg-ratio", "0.10",
]
if IDD_DIR:
    cmd += ["--idd-dir", str(IDD_DIR)]
if SIGN_DIR:
    cmd += ["--traffic-sign-dir", str(SIGN_DIR)]
if RDD_DIR:
    cmd += ["--rdd-dir", str(RDD_DIR)]
if EXTRA_POTHOLE_DIR:
    cmd += ["--extra-pothole-dir", str(EXTRA_POTHOLE_DIR)]

subprocess.run(cmd, check=True)


In [ ]:
# Cell 5: Two-Stage Fine-Tuning for Traffic Sign Model (1280px, fliplr=0.0, IDD Hard Negatives)
if SIGN_DIR:
    !python scripts/finetune_models.py \
        --target traffic_sign \
        --sign-data /kaggle/working/prepared_tuned_datasets/traffic_sign_tuned/traffic_sign_tuned.yaml \
        --sign-weights yolov5/runs/train/traffic_sign_enhanced_1280_pretrained/weights/best.pt \
        --output-dir /kaggle/working/finetune_runs \
        --sign-batch 8 \
        --device 0


In [ ]:
# Cell 6: Two-Stage Fine-Tuning for Road Damage Model (896px, multi-scale, focal loss, anchor_t=5.0)
if RDD_DIR:
    !python scripts/finetune_models.py \
        --target road_damage \
        --rdd-data /kaggle/working/prepared_tuned_datasets/road_damage_tuned/road_damage_tuned.yaml \
        --rdd-weights yolov5/runs/train/enhanced_rdd2022_final/weights/best.pt \
        --output-dir /kaggle/working/finetune_runs \
        --rdd-batch 16 \
        --device 0


In [ ]:
# Cell 7: Package Tuned best.pt Weights & Metrics for One-Click Download
import zipfile
from pathlib import Path

runs_dir = Path("/kaggle/working/finetune_runs")
zip_path = Path("/kaggle/working/tuned_models.zip")

artifacts = [
    ("traffic_sign_tuned_final/weights/best.pt", "traffic_sign_enhanced_1280_pretrained/weights/best.pt"),
    ("traffic_sign_tuned_final/results.csv", "traffic_sign_tuned_results.csv"),
    ("road_damage_tuned_final/weights/best.pt", "enhanced_rdd2022_final/weights/best.pt"),
    ("road_damage_tuned_final/results.csv", "road_damage_tuned_results.csv"),
]

with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    for rel_src, arc_name in artifacts:
        src = runs_dir / rel_src
        if src.exists():
            zf.write(src, arc_name)
            mb = src.stat().st_size / 1e6
            print(f"Added {src} -> {arc_name} ({mb:.2f} MB)")

print(f"\nDone! Download {zip_path} from Kaggle Output and extract into yolov5/runs/train/.")
